# Chapter 4: Descriptive statistics and epidemiological measures

**Data Analysis with Python for Health Specialists** | 3 hours

---

## 1. Central tendency: what is "typical"?

Use the **median** when the distribution is skewed (most lab values, length of stay, health expenditures). Use the **mean** when the distribution is approximately symmetric (height, diastolic BP in healthy adults).

In [ ]:
import pandas as pd
import numpy as np

# Fasting blood glucose for 10 patients (mg/dL)
glucose = pd.Series([92, 98, 104, 95, 88, 110, 97, 340, 101, 93])

print(f"Mean:   {glucose.mean():.1f} mg/dL")
print(f"Median: {glucose.median():.1f} mg/dL")

### Mode

The mode is the most frequent value --- the only measure of central tendency for categorical data.

In [ ]:
diagnoses = pd.Series(["HTN", "T2DM", "HTN", "None", "HTN",
                       "T2DM", "None", "HTN", "COPD", "HTN"])
print(f"Most common diagnosis: {diagnoses.mode()[0]}")
print(f"Frequency: {diagnoses.value_counts().iloc[0]}/{len(diagnoses)}")

## 2. Spread: how variable is the data?

### Standard deviation and variance

In [ ]:
# Blood pressure readings from two clinics
clinic_a = pd.Series([120, 122, 118, 125, 121, 119, 123, 120])
clinic_b = pd.Series([98, 145, 110, 155, 102, 160, 115, 135])

print(f"Clinic A: mean={clinic_a.mean():.1f}, SD={clinic_a.std():.1f}")
print(f"Clinic B: mean={clinic_b.mean():.1f}, SD={clinic_b.std():.1f}")

### IQR and percentiles

The interquartile range (IQR) is the range between the 25th and 75th percentiles. Like the median, it is robust to outliers.

In [ ]:
# Length of hospital stay (days) --- typically right-skewed
los = pd.Series([2, 3, 3, 4, 4, 5, 5, 6, 7, 8, 12, 28, 45])

q25 = los.quantile(0.25)
q75 = los.quantile(0.75)
iqr = q75 - q25

print(f"Median LOS: {los.median():.0f} days")
print(f"IQR: {q25:.0f}--{q75:.0f} days (range: {iqr:.0f})")
print(f"Mean LOS: {los.mean():.1f} days")  # inflated by 28 and 45

### Coefficient of variation

The CV expresses the standard deviation as a percentage of the mean. Useful for comparing variability between measurements with different units.

In [ ]:
# Compare variability of glucose (mg/dL) vs hemoglobin (g/dL)
glucose = pd.Series([95, 102, 88, 110, 97, 105, 92, 98])
hemoglobin = pd.Series([13.2, 14.1, 12.8, 13.5, 14.0, 13.8, 12.5, 13.9])

cv_glucose = (glucose.std() / glucose.mean()) * 100
cv_hb = (hemoglobin.std() / hemoglobin.mean()) * 100

print(f"CV glucose: {cv_glucose:.1f}%")
print(f"CV hemoglobin: {cv_hb:.1f}%")

## 3. Frequency tables and cross-tabulations

In [ ]:
# Load Gapminder data
url = ("https://raw.githubusercontent.com/datasets/"
       "gapminder/main/data/gapminder.csv")
gapminder = pd.read_csv(url)
recent = gapminder[gapminder["year"] == 2007]

# Create life expectancy categories
recent = recent.copy()
recent["le_cat"] = pd.cut(recent["lifeExp"],
                          bins=[0, 55, 70, 85],
                          labels=["Low (<55)", "Medium (55-70)",
                                  "High (>70)"])

# Frequency table
freq = recent["le_cat"].value_counts().sort_index()
print(freq)

# With proportions
freq_table = pd.DataFrame({
    "n": freq,
    "pct": (freq / freq.sum() * 100).round(1),
    "cum_pct": (freq.cumsum() / freq.sum() * 100).round(1)
})
print(freq_table)

In [ ]:
# Cross-tabulate life expectancy category by continent
ct = pd.crosstab(recent["continent"], recent["le_cat"], margins=True)
print(ct)

# With row percentages (percentage within each continent)
ct_pct = pd.crosstab(recent["continent"], recent["le_cat"],
                     normalize="index").round(3) * 100
print(ct_pct)

## 4. Epidemiological measures: counting disease

### Prevalence

Prevalence = Number of existing cases / Total population

In [ ]:
from scipy import stats

# Diabetes prevalence from a health survey
n_total = 5000
n_diabetic = 625

prevalence = n_diabetic / n_total * 100
print(f"Diabetes prevalence: {prevalence:.1f}%")

# With 95% confidence interval (normal approximation)
p = n_diabetic / n_total
se = np.sqrt(p * (1 - p) / n_total)
ci_low = (p - 1.96 * se) * 100
ci_high = (p + 1.96 * se) * 100
print(f"95% CI: [{ci_low:.1f}%, {ci_high:.1f}%]")

### Incidence rate

Incidence rate = Number of new cases / Total person-time at risk

In [ ]:
# Tuberculosis cohort: 200 patients followed for varying durations
new_tb_cases = 18
total_person_years = 850  # sum of follow-up time for all patients

incidence_rate = new_tb_cases / total_person_years
print(f"TB incidence rate: {incidence_rate:.4f} per person-year")
print(f"                 = {incidence_rate * 1000:.1f} per 1,000 person-years")

# 95% CI for incidence rate (Poisson approximation)
ir_se = np.sqrt(new_tb_cases) / total_person_years
ci_low = (incidence_rate - 1.96 * ir_se) * 1000
ci_high = (incidence_rate + 1.96 * ir_se) * 1000
print(f"95% CI: [{ci_low:.1f}, {ci_high:.1f}] per 1,000 person-years")

### Mortality rate and case fatality rate

In [ ]:
# Crude mortality rate
deaths = 1250
population = 500_000
mortality_rate = deaths / population * 100_000
print(f"Crude mortality rate: {mortality_rate:.1f} per 100,000")

# Case fatality rate (CFR)
total_cases = 3200
deaths_among_cases = 128
cfr = deaths_among_cases / total_cases * 100
print(f"Case fatality rate: {cfr:.1f}%")

## 5. Measures of association

### Risk ratio (relative risk)

In [ ]:
import math

# Smoking and lung cancer (hypothetical cohort)
# Exposed (smokers):     a=80 cancer, b=920 no cancer
# Unexposed (non-smokers): c=10 cancer, d=990 no cancer
a, b, c, d = 80, 920, 10, 990

risk_exposed = a / (a + b)
risk_unexposed = c / (c + d)
rr = risk_exposed / risk_unexposed

print(f"Risk in smokers:     {risk_exposed:.3f} ({risk_exposed*100:.1f}%)")
print(f"Risk in non-smokers: {risk_unexposed:.3f} ({risk_unexposed*100:.1f}%)")
print(f"Risk Ratio: {rr:.2f}")

# 95% CI for log(RR)
se_ln_rr = math.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
ln_rr = math.log(rr)
ci_low = math.exp(ln_rr - 1.96 * se_ln_rr)
ci_high = math.exp(ln_rr + 1.96 * se_ln_rr)
print(f"95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

### Odds ratio

In [ ]:
# Case-control study: obesity and knee osteoarthritis
a, b, c, d = 120, 60, 80, 140

odds_ratio = (a * d) / (b * c)
print(f"Odds Ratio: {odds_ratio:.2f}")

# 95% CI
se_ln_or = math.sqrt(1/a + 1/b + 1/c + 1/d)
ln_or = math.log(odds_ratio)
ci_low = math.exp(ln_or - 1.96 * se_ln_or)
ci_high = math.exp(ln_or + 1.96 * se_ln_or)
print(f"95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

### Reusable functions

In [ ]:
def risk_ratio(a, b, c, d, alpha=0.05):
    """Compute risk ratio with confidence interval."""
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)

    rr = (a / (a + b)) / (c / (c + d))
    se = math.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
    ci = (math.exp(math.log(rr) - z * se),
          math.exp(math.log(rr) + z * se))

    return {"RR": round(rr, 3), "CI_low": round(ci[0], 3),
            "CI_high": round(ci[1], 3)}


def odds_ratio_func(a, b, c, d, alpha=0.05):
    """Compute odds ratio with confidence interval."""
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)

    o_r = (a * d) / (b * c)
    se = math.sqrt(1/a + 1/b + 1/c + 1/d)
    ci = (math.exp(math.log(o_r) - z * se),
          math.exp(math.log(o_r) + z * se))

    return {"OR": round(o_r, 3), "CI_low": round(ci[0], 3),
            "CI_high": round(ci[1], 3)}


# Example
print(risk_ratio(80, 920, 10, 990))
print(odds_ratio_func(120, 60, 80, 140))

## 6. Age-standardized rates

Crude rates can be misleading when comparing populations with different age structures. Age standardization removes this confounding.

In [ ]:
# Compare crude vs age-standardized mortality between two regions
data_a = pd.DataFrame({
    "age_group": ["0-14", "15-44", "45-64", "65+"],
    "population": [50000, 80000, 40000, 10000],
    "deaths": [50, 80, 200, 300]
})
data_a["rate"] = data_a["deaths"] / data_a["population"] * 100000

data_b = pd.DataFrame({
    "age_group": ["0-14", "15-44", "45-64", "65+"],
    "population": [15000, 40000, 50000, 45000],
    "deaths": [20, 50, 280, 1500]
})
data_b["rate"] = data_b["deaths"] / data_b["population"] * 100000

# Standard population (e.g., WHO World Standard)
standard_pop = pd.DataFrame({
    "age_group": ["0-14", "15-44", "45-64", "65+"],
    "std_weight": [0.26, 0.42, 0.22, 0.10]
})

# Crude rates
crude_a = data_a["deaths"].sum() / data_a["population"].sum() * 100000
crude_b = data_b["deaths"].sum() / data_b["population"].sum() * 100000
print(f"Crude rate Region A: {crude_a:.1f} per 100,000")
print(f"Crude rate Region B: {crude_b:.1f} per 100,000")

# Age-standardized rates
asr_a = (data_a["rate"] * standard_pop["std_weight"]).sum()
asr_b = (data_b["rate"] * standard_pop["std_weight"]).sum()
print(f"Age-standardized rate Region A: {asr_a:.1f} per 100,000")
print(f"Age-standardized rate Region B: {asr_b:.1f} per 100,000")

## 7. Practical: descriptive statistics with real data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load Gapminder
url = ("https://raw.githubusercontent.com/datasets/"
       "gapminder/main/data/gapminder.csv")
df = pd.read_csv(url)

# Summary statistics by continent (2007)
recent = df[df["year"] == 2007].copy()
summary = recent.groupby("continent")["lifeExp"].agg(
    ["count", "mean", "median", "std",
     lambda x: x.quantile(0.25),
     lambda x: x.quantile(0.75)]
)
summary.columns = ["n", "mean", "median", "sd", "Q1", "Q3"]
print(summary.round(1))

### Computing malaria prevalence by region

In [ ]:
# Simulated malaria data for practice
np.random.seed(42)
countries = (["Nigeria", "DRC", "Mozambique", "Uganda", "Ghana"] * 3 +
             ["India", "Indonesia", "Myanmar", "Pakistan", "Ethiopia"] * 3)

malaria = pd.DataFrame({
    "country": countries,
    "region": ["West Africa", "Central Africa", "East Africa",
               "East Africa", "West Africa"] * 6,
    "year": [2018]*5 + [2019]*5 + [2020]*5 +
            [2018]*5 + [2019]*5 + [2020]*5,
    "population_at_risk": np.random.randint(10_000_000, 200_000_000, 30),
    "estimated_cases": np.random.randint(500_000, 50_000_000, 30)
})

# Compute prevalence per 1,000 population at risk
malaria["prevalence_per_1000"] = (
    malaria["estimated_cases"] / malaria["population_at_risk"] * 1000
)

# Summary by region
region_summary = malaria.groupby("region").agg(
    total_cases=("estimated_cases", "sum"),
    total_pop=("population_at_risk", "sum"),
    n_countries=("country", "nunique")
)
region_summary["prevalence_per_1000"] = (
    region_summary["total_cases"] / region_summary["total_pop"] * 1000
)
print(region_summary.round(1))

# Trend by year
year_trend = malaria.groupby("year").agg(
    total_cases=("estimated_cases", "sum"),
    total_pop=("population_at_risk", "sum")
)
year_trend["prevalence_per_1000"] = (
    year_trend["total_cases"] / year_trend["total_pop"] * 1000
)
print(year_trend.round(1))

## Exercises

1. Compute summary statistics (mean, median, SD, IQR) for life expectancy, GDP per capita, and population for each continent in 2007.
2. Create a cross-tabulation of continent vs life expectancy category (Low/Medium/High). Compute row percentages.
3. A cohort of 10,000 people was followed for 5 years. 450 developed Type 2 diabetes. Compute cumulative incidence and incidence rate (assume 4.5 years average follow-up).
4. In a case-control study of 200 cases and 200 controls, 140 cases and 80 controls were exposed. Compute the odds ratio and 95% CI.
5. Using the age-specific rates above, compute age-standardized rates and check if ranking changes.
6. Compute malaria incidence rate for 5 African countries over 3 years. Which has the highest burden?
7. Write a function `prevalence_ci(cases, total, confidence=0.95)` that returns prevalence and CI. Test with 250/2000.

In [ ]:
# Exercise 1: your code here


In [ ]:
# Exercise 2: your code here


In [ ]:
# Exercise 3: your code here


In [ ]:
# Exercise 4: your code here


In [ ]:
# Exercise 5: your code here


In [ ]:
# Exercise 6: your code here


In [ ]:
# Exercise 7: your code here
